# Step 1: Import Required Libraries

In [1]:
import re
import joblib
import pandas as pd

from transformers import pipeline

C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Observation

- The required libraries were imported successfully.
- Joblib will load the trained model and TF-IDF vectorizer.
- Hugging Face Transformers will provide sentiment analysis for incoming tweets.

# Step 2: Load Saved Model and TF-IDF Vectorizer

In [2]:
model = joblib.load("../models/disaster_classifier.pkl")
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

print("✅ Model loaded successfully!")
print("✅ TF-IDF Vectorizer loaded successfully!")

✅ Model loaded successfully!
✅ TF-IDF Vectorizer loaded successfully!


### Observation

- The trained Logistic Regression model was loaded successfully.
- The saved TF-IDF vectorizer was also loaded successfully.
- These components will be used to classify new tweets without retraining the model.

# Step 3: Load Hugging Face Sentiment Model

In [3]:
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("✅ Hugging Face Sentiment Model Loaded Successfully!")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1896.38it/s]


✅ Hugging Face Sentiment Model Loaded Successfully!


### Observation

- The pre-trained DistilBERT sentiment analysis model was loaded successfully.
- This model predicts whether a tweet expresses Positive or Negative sentiment.
- The predicted sentiment will be combined with TF-IDF features before disaster prediction.

# Step 4: Create Text Cleaning Function

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)        # Remove URLs
    text = re.sub(r"@\w+", "", text)           # Remove mentions
    text = re.sub(r"#", "", text)              # Remove # symbol
    text = re.sub(r"[^a-zA-Z\s]", "", text)    # Remove numbers & special characters
    text = re.sub(r"\s+", " ", text).strip()   # Remove extra spaces
    return text

print("✅ Text cleaning function created successfully!")

✅ Text cleaning function created successfully!


### Observation

- A text preprocessing function was created to clean incoming tweets.
- URLs, mentions, hashtags, numbers, and special characters are removed before prediction.
- This ensures that the input format remains consistent with the data used during model training.

# Step 5: Test Text Cleaning Function

In [5]:
sample_text = "Massive Fire!!! Visit https://abc.com #fire @news"

cleaned_text = clean_text(sample_text)

print("Original Text:")
print(sample_text)

print("\nCleaned Text:")
print(cleaned_text)

Original Text:
Massive Fire!!! Visit https://abc.com #fire @news

Cleaned Text:
massive fire visit fire


### Observation

- The cleaning function successfully removed unwanted characters from the sample tweet.
- The cleaned text contains only meaningful words required for feature extraction.
- This preprocessing step improves prediction consistency and model performance.

# Step 6: Transform Text Using TF-IDF Vectorizer

In [6]:
text_tfidf = tfidf.transform([cleaned_text])

print("TF-IDF Shape:", text_tfidf.shape)

TF-IDF Shape: (1, 5000)


### Observation

- The cleaned tweet was successfully transformed into TF-IDF features.
- The saved TF-IDF vectorizer uses the same vocabulary learned during model training.
- This ensures that new tweets are represented in the same feature space as the training data.

# Step 7: Predict Tweet Sentiment

In [7]:
sentiment_result = sentiment_pipeline(cleaned_text)

print(sentiment_result)

[{'label': 'POSITIVE', 'score': 0.9571012854576111}]


### Observation

- The Hugging Face DistilBERT model predicted the sentiment of the cleaned tweet.
- The output includes both the predicted sentiment label and its confidence score.
- This sentiment information will be used as an additional feature for disaster classification.

# Step 8: Encode Sentiment Feature

In [8]:
sentiment_label = sentiment_result[0]["label"]

sentiment_encoded = 1 if sentiment_label == "POSITIVE" else 0

print("Sentiment:", sentiment_label)
print("Encoded Sentiment:", sentiment_encoded)

Sentiment: POSITIVE
Encoded Sentiment: 1


### Observation

- The predicted sentiment was converted into a numerical feature.
- Positive sentiment is encoded as **1**, while Negative sentiment is encoded as **0**.
- This encoded value will be combined with the TF-IDF features before making the final prediction.

# Step 9: Combine TF-IDF and Sentiment Feature

In [9]:
from scipy.sparse import hstack

final_features = hstack([
    text_tfidf,
    [[sentiment_encoded]]
])

print("Final Feature Shape:", final_features.shape)

Final Feature Shape: (1, 5001)


### Observation

- The TF-IDF features and encoded sentiment feature were successfully combined.
- This final feature vector has the same structure as the data used during model training.
- The combined features are now ready to be passed into the Logistic Regression model for prediction.

# Step 10: Predict Disaster Class

In [10]:
prediction = model.predict(final_features)

print("Prediction:", prediction)

Prediction: [0]


### Observation

- The trained Logistic Regression model generated a prediction for the processed tweet.
- The prediction is returned as a numerical class label.
- The next step will convert this numerical output into a user-friendly result.

# Step 11: Display Final Prediction

In [11]:
if prediction[0] == 1:
    print("🚨 Disaster Tweet")
else:
    print("✅ Non-Disaster Tweet")

✅ Non-Disaster Tweet


### Observation

- The numerical prediction was converted into a human-readable label.
- Tweets predicted as **1** are displayed as **Disaster Tweets**, while **0** indicates **Non-Disaster Tweets**.
- This is the same prediction logic that will be implemented in the Streamlit dashboard.

# Step 12: Test the Model with Multiple Sample Tweets

In [12]:
sample_tweets = [
    "Massive earthquake destroyed several buildings.",
    "I love watching movies with my friends.",
    "A huge wildfire is spreading across the forest.",
    "Had a great lunch today!",
    "Flood warning issued for the nearby villages."
]

for tweet in sample_tweets:
    cleaned = clean_text(tweet)

    sentiment = sentiment_pipeline(cleaned)[0]["label"]
    sentiment_encoded = 1 if sentiment == "POSITIVE" else 0

    tfidf_features = tfidf.transform([cleaned])

    from scipy.sparse import hstack
    final_features = hstack([tfidf_features, [[sentiment_encoded]]])

    prediction = model.predict(final_features)[0]

    label = "🚨 Disaster Tweet" if prediction == 1 else "✅ Non-Disaster Tweet"

    print("=" * 60)
    print("Tweet :", tweet)
    print("Prediction :", label)

Tweet : Massive earthquake destroyed several buildings.
Prediction : 🚨 Disaster Tweet
Tweet : I love watching movies with my friends.
Prediction : ✅ Non-Disaster Tweet
Tweet : A huge wildfire is spreading across the forest.
Prediction : 🚨 Disaster Tweet
Tweet : Had a great lunch today!
Prediction : ✅ Non-Disaster Tweet
Tweet : Flood warning issued for the nearby villages.
Prediction : 🚨 Disaster Tweet


### Observation

- The trained model was tested on multiple sample tweets.
- The model successfully classified both disaster-related and non-disaster tweets.
- This confirms that the complete prediction pipeline works correctly before deployment in the Streamlit application.